In [ ]:
#Limpeza da tabela campeonato-brasileiro-gols.csv
 
# Esse script segue uma ordem lógica de limpeza de dados:
#1. Carregar
#2. Tratar valores nulos "disfarçados"
#3. Limpar texto (espaços, maiúsculas/minúsculas)
#4. Correção do enconding do arquivo
#5. padronização do tipo de gol.
#6. Correção da coluna minuto.
#7. remover duplicatas
#8. Ordenação da tabela
#9. Corrigir números
#10. Salvar



In [ ]:
import pandas as pd
import numpy as np

#carregamento de dados brutos e bagunçados
df = pd.read_csv(r"D:\projetos git\analise-de-times-br\data\raw\campeonato-brasileiro-gols.csv")
df.head(10)


In [ ]:
#limpar nulos disfarçados
fake_nulls = ["", " ", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "none"]
df.replace(fake_nulls, np.nan, inplace=True)

In [ ]:
#limpar texto de maisculas e minisculas e limpar espaços em branco
coluna_atleta = df['atleta']
coluna_clube = df['clube']
for coluna in coluna_atleta:
    df['atleta'] = df['atleta'].str.strip().str.title()
for coluna in coluna_clube:
    df['clube'] = df['clube'].str.strip().str.title() 

       

In [ ]:
#Correção do encoding do arquivo, o arquivo original esta com enconding incorreto
#Se faz necessário corrigir o encoding para evitar problemas com caracteres especiais
def corrigur_encoding(valor):
    if not isinstance(valor, str):
        return valor
    try:
        return valor.encode('latin1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return valor

for coluna in ['atleta', 'clube']:
    df[coluna] = df[coluna].apply(corrigur_encoding)

In [ ]:
#padronizacao do tipo de gol, a tabela apresnetava diversos formatos de marcação de gol,
#Entao foi cirado um dicionario para padronizar os valores e evitar perdas.
mapa_tipo_de_gol = {
    "Penalty": "Pênalti", "penalty": "Pênalti", "PENALTY": "Pênalti",
    "penalti": "Pênalti", "Pênalti": "Pênalti", "PÊNALTI": "Pênalti",
    "Contra": "Contra", "autogol": "Contra", "Gol Contra": "Contra",
    "GOL CONTRA": "Contra", "gol contra": "Contra",
}

df['tipo_de_gol'] = df['tipo_de_gol'].map(mapa_tipo_de_gol)
df['tipo_de_gol'] = df['tipo_de_gol'].fillna('Normal')

In [ ]:
#correção da coluna minuto, a coluna minuto apresntava situacoes de acréscimo, como "45+2" 
# para corrigir isso precisamos serparar o minuto do valor do acrescimo e somar o valores 
# para posteriomente podermos trabalhar com esses valores.

#A correção de outliers tambem foi feita ja que as partidas de futebol nao duram mais de 120 minutos, entao quqalquer valor sera subsituido por NaN.
df["minuto"] = df["minuto"].astype(object)
acrescimo = df["minuto"].astype(str).str.contains("\+", na=False)

partes = df.loc[acrescimo, "minuto"].str.split("+", expand=True)
df.loc[acrescimo,"minuto"] = (pd.to_numeric(partes[0], errors='coerce') + pd.to_numeric(partes[1], errors='coerce'))

df["minuto"] = pd.to_numeric(df["minuto"], errors='coerce')

limite_maximo_minutos = 100
outliers = df["minuto"] > limite_maximo_minutos
df.loc[outliers, "minuto"] = np.nan

df["minuto"] = df["minuto"].astype("Int64")

In [ ]:
#remover duplicatas
chave_unica = ["partida_id", "atleta", "minuto", "clube"]
duplicatas = df.duplicated(subset=chave_unica, keep='first')


In [ ]:
#ordenação das colunas para facilitar a leitura e analise dos dados
ordem_das_colunas = ["partida_id", "rodata", "minuto", "clube","atleta", "tipo_de_gol"]
df = df[ordem_das_colunas]

df = df.sort_values(by=["partida_id", "rodata"]).reset_index(drop=True)

In [ ]:
#salvar arquivo limpo
print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())
print("\nDistribuição de tipo_de_gol:")
print(df["tipo_de_gol"].value_counts())
 
caminho_saida = r"D:\projetos git\analise-de-times-br\data\processed\campeonato-brasileiro-gols-limpo.csv"
df.to_csv(caminho_saida, index=False)
display(df)


# OBS: Essa tabela só cobre partida_id de 4607 em diante
# (partidas a partir de 19/04/2014, segundo a tabela full). Isso é uma
# característica do dataset original do Kaggle, não uma perda causada
# pela limpeza.